[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 中等: Top-k / Top-p (Nucleus) 采样

实现**带 top-k 和 top-p 过滤的采样**——标准的 LLM 解码策略。

### 函数签名
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) 未归一化的 log 概率
    # 返回: 采样的 token 索引
```

### 算法
1. 温度缩放: `logits /= temperature`
2. Top-k: 只保留 top-k 个 logits，其余设为 `-inf`
3. Top-p: 按概率排序，掩码掉累积概率超过 p 的 token
4. 从过滤后的分布中采样

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✏️ 在此实现你的代码

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    pass  # 温度缩放, top-k 过滤, top-p 过滤, 采样

torch.multinomial 是 PyTorch 中用于从多项分布中抽取样本的函数。它的核心作用是根据给定的权重（概率）对索引进行有放回或无放回的随机采样。

```python
torch.multinomial(input, num_samples, replacement=False, *, generator=None, out=None)
```
- input：一个二维张量，表示概率分布。每一行代表一个概率分布，每一列代表一个概率。
- num_samples：一个整数，表示从概率分布中抽取的样本数量。
- replacement：一个布尔值，表示是否允许重复抽取样本。如果为 True，则允许重复抽取样本；如果为 False，则不允许重复抽取样本。

---
torch.randint 是 PyTorch 中用于生成一个指定范围内的随机整数张量的函数。

```python
torch.randint(low=0, high, size, *, generator=None, out=None, dtype=None, layout=torch.strided, device=None, requires_grad=False)
```

- low：一个整数，表示生成随机整数的最小值。
- high：一个整数，表示生成随机整数的最大值。
- size：一个元组，表示生成张量的形状。

In [ ]:
import torch.nn.functional as F

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    '''
    基于 Top-K 和 Top-P 的采样

    Args:
        logits: (V,) 未归一化的对数概率分布
        top_k: Top-K 采样的 k
        top_p: Top-P 采样的 p
        temperature: 温度参数

    Returns:
        int: 采样的索引
    '''
    
    if temperature != 1.0:
        logits = logits / temperature
    
    if top_k > 0:
        # 获取 top_k 个最大的 logits 的值和索引
        top_k_values, top_k_indices = torch.topk(logits, top_k) # [top_k, ] [top_k,]
        # 创建掩码，只保留 top_k 位置
        mask = torch.zeros_like(logits, dtype=torch.bool)
        mask[top_k_indices] = True
        # 将非 top_k 的位置设为 -inf
        logits = logits.masked_fill(~mask, float('-inf'))
    
    probs = F.softmax(logits, dim=-1) # 将 top_k 的概率归一化，并且将对数概率转成概率
    
    if top_p < 1.0:
        # 按概率从高到低排序
        sorted_probs, sorted_indices = torch.sort(probs, descending=True) # [top_k,] [top_k,]
        # 计算累积概率
        cumsum_probs = torch.cumsum(sorted_probs, dim=-1)
        # 找到累积概率超过 top_p 的位置
        # 保留累积概率 <= top_p 的 token，确保最大概率的 token 保留
        mask = cumsum_probs <= top_p
        mask[0] = True
        # 根据索引构建过滤掩码，将累积概率 <= top_p 的 token 保留,其他置 0
        sorted_mask = torch.zeros_like(probs, dtype=torch.bool)
        sorted_mask[sorted_indices[mask]] = True
        probs = probs.masked_fill(~sorted_mask, 0.0)
    
    # 重新归一化（因为 top-p 过滤后概率和可能不为 1）
    if top_p < 1.0:
        probs = probs / probs.sum()
    
    # 如果所有概率为 0 或包含 nan，回退到均匀采样，根据权重无放回的随机采样
    if torch.isnan(probs).any() or (probs == 0).all():
        return torch.randint(0, len(logits), (1,)).item()
    
    return torch.multinomial(probs, 1).item()

In [ ]:
# 🧪 调试
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

In [ ]:
# ✅ 提交
from torch_judge import check
check('topk_sampling')